[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/main/course/7_juegos/_assets/01_decidir_en_el_arbol_t.ipynb)

# Notebook · Decidir en el árbol T

Cinco algoritmos sobre el mismo árbol de juguete: **minimax**,
**expectiminimax**, **alfa-beta**, **minimax con corte** y
**profundización iterativa**. El código es el Python de las páginas, con su
número de línea `# (n)`.

- Corre las celdas en orden con `Shift + Enter`.
- Mueve el deslizador **paso** (o dale ▶): ves el árbol, la línea que corre y
  sus variables.
- Al final de cada sección hay un **Pruébalo**: cambia un número y vuelve a
  correr.

## 1 · Preparación

La maquinaria: 23 celdas cortas. No hace falta leerlas: corre y sigue. Las que
dibujan están plegadas en Colab.

In [ ]:
# === Celda 1 · Bibliotecas ======================================
import matplotlib.pyplot as plt

try:
    from ipywidgets import (HBox, IntSlider, Play, VBox,
                            interactive_output, jslink)
    from IPython.display import display
    HAY_WIDGETS = True
except ImportError:          # sin widgets: se imprime con un for
    HAY_WIDGETS = False
print("listo · widgets:", HAY_WIDGETS)

In [ ]:
# === Celda 2 · El árbol T =======================================
from fractions import Fraction
from math import inf

# R (MAX) → I, C, D (MIN) por izq, centro, der;
# C → C1, C2 (MAX) por c1, c2. Los números son hojas.
ARBOL_T = ((3, 6), ((5, 2), (7, 8)), (2, 12))
ARBOL_T1 = ((3, 6), (2, 12))            # T sin el centro
EVAL_T = {"I": 5, "C": 4, "D": 7, "C1": 6, "C2": 9}
JUGADAS_R = {3: ["izq", "centro", "der"], 2: ["izq", "der"]}
NOMBRE_R = {"izq": "I", "centro": "C", "der": "D"}

In [ ]:
# === Celda 3 · Nombrar los nodos ================================
def armar(arbol, s="R", k=0, red=None):
    # {nodo: {jugada: hijo}}, {hoja: número}, {nodo: profundidad}
    hijos, valor, prof = red or ({}, {}, {})
    prof[s] = k
    if not isinstance(arbol, tuple):     # una hoja: «D-2»
        valor[s] = arbol
        return hijos, valor, prof
    hijos[s] = {}
    for i, h in enumerate(arbol, 1):
        a = JUGADAS_R[len(arbol)][i - 1] if k == 0 else f"{s.lower()}{i}"
        sep = "" if isinstance(h, tuple) else "-"
        hijos[s][a] = n = NOMBRE_R[a] if k == 0 else f"{s}{sep}{i}"
        armar(h, n, k + 1, (hijos, valor, prof))
    return hijos, valor, prof

In [ ]:
# === Celda 4 · Las reglas: S_F, U, Pl, A y Pr ===================
def usar(arbol, azar=()):
    """Elige el árbol, y qué nodos son de azar."""
    global HIJOS, VALOR, PROF, AZAR
    HIJOS, VALOR, PROF = armar(arbol)
    AZAR = set(azar)

def es_final(s): return s in VALOR
def utilidad(s): return VALOR[s]
def u_final(s): return Fraction(VALOR[s], 100)   # la U de la clase 3
def acciones(s): return list(HIJOS[s])
def pr(s, a): return Fraction(1, len(HIJOS[s]))
def pl(s):
    return "AZAR" if s in AZAR else ("MAX", "MIN")[PROF[s] % 2]

In [ ]:
# === Celda 5 · T y EVAL, con una cuenta =========================
# Generar un nodo es llamar a transicion: la cuenta lo anota.
CUENTA = dict(generados=["R"], total=0, evaluados=set(), terminadas=0)

def transicion(s, a):
    hijo = HIJOS[s][a]
    CUENTA["generados"].append(hijo)
    CUENTA["total"] += 1
    return hijo

def evaluar(s):                        # EVAL: una estimación de s
    CUENTA["evaluados"].add(s)
    return EVAL_T[s]

usar(ARBOL_T)

In [ ]:
# === Celda 6 · paso: una foto de las variables ==================
VISIBLES = ("s", "h", "x", "pila", "v", "w", "alfa", "beta",
            "mejor_valor", "mejor_jugada", "jugada", "d", "corta")

def nueva_busqueda(s):                 # la cuenta vuelve a empezar en s
    CUENTA.update(generados=[s], evaluados=set(), total=CUENTA["total"] + 1)

def paso(evento, linea, variables):
    """Una fila de la traza: qué pasó, qué línea y las variables."""
    if evento == "fin":
        CUENTA["terminadas"] += 1        # para el reloj de la sección 7
    foto = {k: variables[k] for k in VISIBLES if k in variables}
    return dict(evento=evento, linea=linea, vars=foto,
                generados=list(CUENTA["generados"]),
                evaluados=set(CUENTA["evaluados"]))

In [ ]:
# === Celda 7 · Lo que un corte deja fuera =======================
def podar(s, a, pila, corta):          # las filas ✗ de la página
    for b in acciones(s)[acciones(s).index(a) + 1:]:
        x = HIJOS[s][b]                # solo se mira el nombre
        yield paso("podado", "", locals())

def bajo(n):                           # n y sus descendientes
    return [n] + [m for h in HIJOS.get(n, {}).values() for m in bajo(h)]

In [ ]:
# === Celda 8 · Las filas de R ==================================
CARGA = ("mejor_jugada", "jugada", "d", "alfa")

def de_r(f, x, arriba):
    """En R: α viene de la raíz, β = +∞ y v es mejor_valor (o α)."""
    arriba.update({k: x[k] for k in CARGA if k in x})
    f["alfa"] = arriba.get("alfa")
    f["beta"] = None if f["alfa"] is None else inf
    f["v"] = x.get("mejor_valor", f["alfa"])

In [ ]:
# === Celda 9 · De pasos a la tabla de la página =================
def fila_de(n, p, arriba):
    x, ev = p["vars"], p["evento"]
    f = dict(n=n, evento=ev, linea=p["linea"], nodo=x.get("x", x["s"]),
             pila=x.get("pila", ["R"]), v=x.get("v"), w=None, hijo=None,
             alfa=x.get("alfa"), beta=x.get("beta"), corta=x.get("corta"),
             hechos=p["generados"], evaluados=p["evaluados"])
    if "pila" not in x:                # una fila de R
        de_r(f, x, arriba)
    if ev in ("regresa", "raiz", "corte beta", "corte alfa"):
        f.update(w=x["w"], hijo=x["h"], corta=(ev.split() + [None])[1])
    if ev == "podado":
        f["w"] = VALOR.get(f["nodo"])
    return dict(f, **{k: arriba.get(k) for k in CARGA[:3]})

In [ ]:
# === Celda 10 · correr un algoritmo =============================
def correr(algoritmo):
    """Corre el generador. Devuelve (filas, lo que devolvió)."""
    CUENTA.update(generados=["R"], total=0, evaluados=set(), terminadas=0)
    pasos = []
    while True:
        try:
            pasos.append(next(algoritmo))
        except StopIteration as fin:
            resultado = fin.value
            break
    arriba = {}
    filas = [fila_de(n, p, arriba) for n, p in enumerate(pasos, 1)]
    return filas, resultado

In [ ]:
# === Celda 11 · Lo que se recuerda de cada nodo ================
def recordar(f, m):
    if f["evento"] in ("antes", "inicio"):   # empieza una búsqueda
        m.update(v={}, ab={}, dev={}, podados=set(), cotas=set())
    if f["evento"] != "podado":
        m["v"][f["nodo"]] = f["v"]
        m["ab"][f["nodo"]] = (f["alfa"], f["beta"])
    if f["hijo"] is not None:
        m["dev"][f["hijo"]] = f["w"]
    if f["corta"] and f["evento"] != "podado":
        fuera = [h for h in HIJOS[f["nodo"]].values()
                 if h not in f["hechos"]]
        m["podados"].update(y for h in fuera for y in bajo(h))
        m["cotas"].update([f["nodo"]] if fuera else [])

In [ ]:
# === Celda 12 · El estado de un nodo ===========================
def estado_de(n, f, m):
    e = dict(estado="pormirar", v=None, alfa=None, beta=None, cota=False)
    abiertos = () if f["evento"] in ("fin", "entrega") else f["pila"]
    if n in abiertos:
        a, b = m["ab"].get(n, (None, None))
        e.update(estado="pila", v=m["v"].get(n), alfa=a, beta=b)
    elif n in f["hechos"]:
        v = VALOR[n] if n in VALOR else m["dev"].get(n, m["v"].get(n))
        ev = "evaluado" if n in f["evaluados"] else "devuelto"
        e.update(estado=ev, v=v, cota=n in m["cotas"])
    elif n in m["podados"]:
        e["estado"] = "podado"
    return e

In [ ]:
# === Celda 13 · Fotos y números =================================
def fotos_de(filas):                   # el estado de todos, fila por fila
    m = {}
    return [recordar(f, m) or {n: estado_de(n, f, m) for n in PROF}
            for f in filas]

def fmt(x):
    return "" if x is None else {inf: "+∞", -inf: "−∞"}.get(x, str(x))

def tabla(filas):
    for f in filas:
        print(renglon(f))

In [ ]:
# === Celda 14 · Imprimir una fila ===============================
def renglon(f):
    """# · línea · pila · (α, β) · v · w · ¿corta? · mejor_jugada"""
    if f["evento"] == "podado":
        return (f"{f['n']:>2} ✗ {f['nodo']} ({fmt(f['w'])}) no se genera:"
                f" corte {f['corta']}")
    ab = "" if f["alfa"] is None else f"({fmt(f['alfa'])}, {fmt(f['beta'])})"
    w = fmt(f["w"]) + (f" ({f['hijo']})" if f["hijo"] and "-" not in
                       f["hijo"] else "")
    return (f"{f['n']:>2} │ {f['linea']:<5} │ {'›'.join(f['pila']):<8} │ "
            f"{ab:<9} │ v {fmt(f['v']):<4} │ w {w:<8} │ "
            f"{f['corta'] or '':<4} │ {f['mejor_jugada'] or 'ninguna'}"
            f" │ gen {len(f['hechos'])}")

In [ ]:
# === Celda 15 · Los números de línea ===========================
import inspect
import re

def rango(linea):                      # «12–15» → [12, 13, 14, 15]
    a, _, b = linea.partition("–")
    return list(range(int(a), int(b or a) + 1)) if a else []

def marcas(fn):
    return {int(n) for n in re.findall(r"# \((\d+)\)",
                                       inspect.getsource(fn))}

In [ ]:
# === Celda 16 · El código con la línea que corre ================
def codigo(f, fuentes):
    """La función que corre, sin las fotos; ▶ en la línea del paso."""
    ls = rango(f["linea"])
    fn = max(fuentes, key=lambda g: len(marcas(g) & set(ls)))
    return "\n".join(con_flechas(inspect.getsource(fn), ls))

In [ ]:
# === Celda 17 · Las flechas ▶ ===================================
def con_flechas(fuente, ls):
    sangria = None                     # la de la última línea activa
    for r in fuente.splitlines():
        if "paso(" in r or "podar(" in r:
            continue                   # instrumentación: no se muestra
        m = re.search(r"# \((\d+)\)", r)
        ind = len(r) - len(r.lstrip())
        if m:
            sangria = ind if int(m.group(1)) in ls else None
        elif sangria is not None and ind <= sangria:
            sangria = None
        yield ("▶ " if sangria is not None else "  ") + r

In [ ]:
# === Celda 18 · Posiciones del dibujo ===========================
def tablero():                         # una copia del árbol en uso
    return dict(hijos=HIJOS, valor=VALOR, prof=PROF, azar=set(AZAR))

def posiciones(t):
    """(x, y) de cada nodo: hojas en fila; un padre, al centro."""
    pos, hojas = {}, iter(range(99))
    def colocar(n):
        xs = [colocar(h) for h in t["hijos"].get(n, {}).values()]
        xs = xs or [next(hojas)]
        pos[n] = (sum(xs) / len(xs), -t["prof"][n])
        return pos[n][0]
    colocar("R")
    return pos

In [ ]:
# @title Celda 19 · Colores y formas
COLOR = {"pormirar": "#eeeeee", "pila": "#ffcf5c", "devuelto": "#8ecae6",
         "evaluado": "#a7d7a0", "podado": "#ffffff"}
MARCA = {"MAX": "^", "MIN": "v", "AZAR": "o", "HOJA": "s"}
LEYENDA = ("▲ MAX  ▼ MIN  ● azar  ■ hoja  ·  amarillo: en la pila  ·  "
           "azul: devuelto  ·  verde: EVAL  ·  punteado ?: podado")

def etiquetas(ax, n, x, y, e, k):      # nombre, (α, β) y «cota»
    if k != "HOJA":
        ax.text(x, y + .4, n, ha="center", fontsize=9, color="#333")
    if e["alfa"] is not None and e["estado"] == "pila":
        ax.text(x + .3, y, f"({fmt(e['alfa'])}, {fmt(e['beta'])})",
                fontsize=9, color="#d1495b", va="center")
    if e["cota"]:
        ax.text(x, y - .42, "cota", ha="center", fontsize=8, c="#d1495b")

In [ ]:
# @title Celda 20 · Dibujar un nodo
def nodo(ax, n, x, y, e, t):
    k = "HOJA" if n in t["valor"] else "AZAR" if n in t["azar"] else (
        "MAX", "MIN")[t["prof"][n] % 2]
    est, pila = e["estado"], e["estado"] == "pila"
    ax.scatter(x, y, s=1300, marker=MARCA[k], c=COLOR[est], zorder=2,
               edgecolors="#d1495b" if pila else "#444",
               linewidths=3 if pila else 1.2,
               linestyle="--" if est == "podado" else "-")
    txt = "?" if est == "podado" else fmt(t["valor"].get(n, e["v"]))
    ax.text(x, y + {"^": -.06, "v": .06}.get(MARCA[k], 0), txt, zorder=3,
            ha="center", va="center", fontsize=10, weight="bold",
            color="#999" if est == "pormirar" else "#111")
    return k

In [ ]:
# @title Celda 21 · Dibujar el árbol
def dibujar(foto, t, ax, titulo=""):
    pos = posiciones(t)
    for p, hs in t["hijos"].items():
        for h in hs.values():
            ls = ":" if foto[h]["estado"] == "podado" else "-"
            ax.plot(*zip(pos[p], pos[h]), ls=ls, c="#888", zorder=1)
    for n, (x, y) in pos.items():
        k = nodo(ax, n, x, y, foto[n], t)
        etiquetas(ax, n, x, y, foto[n], k)
    ax.set_ylim(min(y for _, y in pos.values()) - .55, .8)
    ax.set_title(titulo, loc="left", fontsize=10)
    ax.axis("off")

In [ ]:
# @title Celda 22 · Mostrar un paso
def mostrar(filas, fotos, t, fuentes, k):
    f = filas[k - 1]
    fig, ax = plt.subplots(figsize=(9, 4.6))
    titulo = (f"paso {k} de {len(filas)} · línea {f['linea'] or '—'} · "
              f"generados {len(f['hechos'])}/{len(t['prof'])} · "
              f"pila {' › '.join(f['pila'])}")
    dibujar(fotos[k - 1], t, ax, titulo)
    fig.text(.01, .01, LEYENDA, fontsize=8, color="#555")
    plt.show()
    print(renglon(f), "\n")
    print(codigo(f, fuentes) if f["linea"] else "(fila ✗: no corre nada)")

In [ ]:
# @title Celda 23 · ver: el deslizador de pasos
def ver(filas, *fuentes):
    t, fotos = tablero(), fotos_de(filas)
    if not HAY_WIDGETS:                # respaldo: todos los pasos
        tabla(filas)
        return mostrar(filas, fotos, t, fuentes, len(filas))
    barra = IntSlider(value=1, min=1, max=len(filas), description="paso")
    play = Play(value=1, min=1, max=len(filas), interval=1800)
    jslink((play, "value"), (barra, "value"))
    vista = interactive_output(
        lambda k: mostrar(filas, fotos, t, fuentes, k), {"k": barra})
    display(VBox([HBox([play, barra]), vista]))

def lupa(filas, k, *fuentes):          # un solo paso, fijo
    mostrar(filas, fotos_de(filas), tablero(), fuentes, k)

## 2 · El árbol T

R es la raíz; mueve MAX. Sus jugadas `izq`, `centro` y `der` llevan a I, C
y D. De C salen `c1` y `c2` a C1 y C2. Los números son finales: su $U$.

In [ ]:
usar(ARBOL_T)
fig, ax = plt.subplots(figsize=(9, 4.2))
vacio = dict(estado="pormirar", v=None, alfa=None, beta=None, cota=False)
dibujar({n: vacio for n in PROF}, tablero(), ax, "el árbol T")
plt.show()
print("A(R) =", acciones("R"), "· A(C) =", acciones("C"))
print("T(C, c2) =", HIJOS["C"]["c2"], "· Pl(C2) =", pl("C2"))

**Pregunta.** ¿Quién mueve en cada nivel? (▲ es MAX, ▼ es MIN.)

<details><summary>Respuesta</summary>

R mueve MAX; en I, C y D mueve MIN; en C1 y C2 otra vez MAX. Los turnos
alternan. Las hojas son finales: nadie mueve.

</details>

**Cómo leer el código de cada algoritmo.** Es el Python de la página, con
su `# (n)`. Hay dos añadidos: `yield paso(...)` toma una foto de las
variables (no cambia nada) y `yield from` deja pasar las fotos de los
hijos. `pila` es la lista de quienes esperan respuesta.

## 3 · Minimax

MAX se queda con el mayor $w$ de sus hijos; MIN, con el menor. La raíz no
devuelve un valor: devuelve **la jugada**.

In [ ]:
def decidir_minimax(s):                  # (1)
    mejor_valor = -inf                   # (2) nada visto
    mejor_jugada = None                  # (2)
    yield paso("inicio", "2", locals())
    for a in acciones(s):                # (3)
        h = transicion(s, a)             # (4) genera el hijo
        w = yield from minimax(h, [s, h])    # (4) cuánto vale a
        if w > mejor_valor:              # (5) estricto
            mejor_valor = w
            mejor_jugada = a
        yield paso("raiz", "4–5", locals())
    yield paso("fin", "6", locals())
    return mejor_jugada                  # (6) la jugada, no w

In [ ]:
def minimax(s, pila):                    # (7) devuelve V(s)
    if es_final(s):                      # (8) final: su U
        return utilidad(s)
    if pl(s) == "MAX":                   # (9)
        return (yield from minimax_max(s, pila))
    else:                                # (15) Pl(s) = MIN
        return (yield from minimax_min(s, pila))

Las dos ramas son espejo: cambian $-\infty$ por $+\infty$ y `max` por
`min`.

In [ ]:
def minimax_max(s, pila):                # la rama MAX
    v = -inf                             # (10) menor que todo
    yield paso("entra", "10", locals())
    for a in acciones(s):                # (11)
        h = transicion(s, a)             # (12)
        w = yield from minimax(h, pila + [h])    # (12) el hijo da w
        v = max(v, w)                    # (13) el mayor
        yield paso("regresa", "12–13", locals())
    return v                             # (14)

In [ ]:
def minimax_min(s, pila):                # la rama MIN
    v = inf                              # (16) mayor que todo
    yield paso("entra", "16", locals())
    for a in acciones(s):                # (17)
        h = transicion(s, a)             # (18)
        w = yield from minimax(h, pila + [h])    # (18) el hijo da w
        v = min(v, w)                    # (19) el menor
        yield paso("regresa", "18–19", locals())
    return v                             # (20)

Ahora sí: córrelo y mueve el deslizador. Cada paso es una fila de la tabla
de la página.

In [ ]:
usar(ARBOL_T)
filas_mm, jugada = correr(decidir_minimax("R"))
print("jugada:", jugada, "· generados:", len(filas_mm[-1]["hechos"]), "de 14")
ver(filas_mm, decidir_minimax, minimax, minimax_max, minimax_min)

**Pruébalo.** Cambia la primera hoja de D, el 2, por un 9, y corre.

<details><summary>Qué pasa</summary>

D vale $\min(9, 12) = 9$, más que el 5 de C: sale `der`. Una sola hoja
cambió la jugada, porque minimax mira todo.

</details>

In [ ]:
prueba = ((3, 6), ((5, 2), (7, 8)), (2, 12))     # cambia un número
usar(prueba)
print("jugada:", correr(decidir_minimax("R"))[1])
usar(ARBOL_T)

## 4 · Cuando decide un dado: expectiminimax

Ahora I, C y D no son de MIN: son **volados parejos**, cada hijo con
probabilidad ½. Un nodo de azar no elige: suma `Pr · w` de todos sus hijos.
La tabla sigue solo a los nodos de azar; C1 y C2 se valoran por dentro.

In [ ]:
def decidir_expectiminimax(s):           # (1)
    mejor_valor = -inf                   # (2)
    mejor_jugada = None                  # (2)
    yield paso("inicio", "2", locals())
    for a in acciones(s):                # (3)
        h = transicion(s, a)             # (4)
        w = yield from expectiminimax(h, [s, h])    # (4) en promedio
        if w > mejor_valor:              # (5) estricto
            mejor_valor = w
            mejor_jugada = a
        yield paso("raiz", "4–5", locals())
    yield paso("fin", "6", locals())
    return mejor_jugada                  # (6) jugada

In [ ]:
def expectiminimax(s, pila):             # (7)
    if es_final(s):                      # (8) su U
        return utilidad(s)
    if pl(s) == "MAX":                   # (9)
        return (yield from em_max(s, pila))
    elif pl(s) == "MIN":                 # (15)
        return (yield from em_min(s, pila))
    else:                                # (21) AZAR
        return (yield from em_azar(s, pila))

In [ ]:
def em_max(s, pila):                     # MAX, sin filas propias
    v = -inf                             # (10)
    for a in acciones(s):                # (11)
        h = transicion(s, a)             # (12)
        w = yield from expectiminimax(h, pila + [h])   # (12)
        v = max(v, w)                    # (13) mayor
    return v                             # (14)

def em_min(s, pila):                     # MIN, sin filas propias
    v = inf                              # (16)
    for a in acciones(s):                # (17)
        h = transicion(s, a)             # (18)
        w = yield from expectiminimax(h, pila + [h])   # (18)
        v = min(v, w)                    # (19) menor
    return v                             # (20)

In [ ]:
def em_azar(s, pila):                    # la rama AZAR
    v = 0                                # (22) suma: empieza en 0
    yield paso("entra", "22", locals())
    for a in acciones(s):                # (23) todos los hijos
        h = transicion(s, a)             # (24)
        w = yield from expectiminimax(h, pila + [h])   # (24)
        v = v + pr(s, a) * w             # (25) pesa su Pr
        yield paso("regresa", "24–25", locals())
    return v                             # (26) valor esperado

In [ ]:
usar(ARBOL_T, azar=("I", "C", "D"))
filas_az, jugada = correr(decidir_expectiminimax("R"))
for f in filas_az:
    if f["evento"] == "raiz":
        print(f"{f['hijo']} vale {fmt(f['w'])}")
print("jugada:", jugada, "  (minimax decía centro)")
ver(filas_az, decidir_expectiminimax, expectiminimax, em_azar)

**Pruébalo.** Cambia el 12 de D por un 4: ¿sigue ganando `der`?

<details><summary>Qué pasa</summary>

D vale $\tfrac12\cdot 2 + \tfrac12\cdot 4 = 3$, menos que $13/2$: gana
`centro`. Con azar cuenta cada hoja, no solo la peor.

</details>

In [ ]:
prueba = ((3, 6), ((5, 2), (7, 8)), (2, 12))     # cambia el 12
usar(prueba, azar=("I", "C", "D"))
print("jugada:", correr(decidir_expectiminimax("R"))[1])
usar(ARBOL_T)

## 5 · Alfa-beta

- **α**: lo que MAX ya tiene asegurado en el camino. Solo sube.
- **β**: lo que MIN ya tiene asegurado en el camino. Solo baja.

La regla: **MAX compara v ≥ β; MIN compara v ≤ α**. Cada uno mira el número
del rival, que heredó de arriba. Si se cumple, corta: el rival nunca
dejaría llegar la partida ahí.

In [ ]:
def decidir_alfa_beta(s):                # (1)
    alfa = -inf                          # (2) hace de mejor_valor
    mejor_jugada = None                  # (2)
    yield paso("inicio", "2", locals())
    for a in acciones(s):                # (3)
        h = transicion(s, a)             # (4) pasa su α; β = +∞
        w = yield from alfa_beta(h, alfa, inf, [s, h])    # (4)
        if w > alfa:                     # (5) estricto
            alfa = w
            mejor_jugada = a
        yield paso("raiz", "4–5", locals())
    yield paso("fin", "6", locals())
    return mejor_jugada                  # (6)

In [ ]:
def alfa_beta(s, alfa, beta, pila):      # (7) devuelve v
    if es_final(s):                      # (8)
        return utilidad(s)
    if pl(s) == "MAX":                   # (9)
        return (yield from ab_max(s, alfa, beta, pila))
    else:                                # (17)
        return (yield from ab_min(s, alfa, beta, pila))

In [ ]:
def ab_max(s, alfa, beta, pila):         # la rama MAX
    v = -inf                             # (10)
    yield paso("entra", "10", locals())
    for a in acciones(s):                # (11)
        h = transicion(s, a)             # (12) hereda la ventana
        w = yield from alfa_beta(h, alfa, beta, pila + [h])  # (12)
        v = max(v, w)                    # (13)
        if v >= beta:                    # (14) lee el β del rival
            yield paso("corte beta", "12–14", locals())
            yield from podar(s, a, pila, "beta")
            return v                     # (14) corte beta
        alfa = max(alfa, v)              # (15) sube su α
        yield paso("regresa", "12–15", locals())
    return v                             # (16) valor o cota

In [ ]:
def ab_min(s, alfa, beta, pila):         # la rama MIN
    v = inf                              # (18)
    yield paso("entra", "18", locals())
    for a in acciones(s):                # (19)
        h = transicion(s, a)             # (20) hereda la ventana
        w = yield from alfa_beta(h, alfa, beta, pila + [h])  # (20)
        v = min(v, w)                    # (21)
        if v <= alfa:                    # (22) lee el α del rival
            yield paso("corte alfa", "20–22", locals())
            yield from podar(s, a, pila, "alfa")
            return v                     # (22) corte alfa
        beta = min(beta, v)              # (23) baja su β
        yield paso("regresa", "20–23", locals())
    return v                             # (24) valor o cota

Primero el árbol chico, T1 (T sin el centro): 6 de 7 nodos.

In [ ]:
AB = (decidir_alfa_beta, alfa_beta, ab_max, ab_min)
usar(ARBOL_T1)
filas_t1, jugada = correr(decidir_alfa_beta("R"))
print("jugada:", jugada, "· generados:", len(filas_t1[-1]["hechos"]), "de 7")
ver(filas_t1, *AB)

Ahora T: 12 de 14. Minimax generaba los 14; la jugada es la misma.

In [ ]:
usar(ARBOL_T)
filas_ab, jugada = correr(decidir_alfa_beta("R"))
print("jugada:", jugada, "· generados:", len(filas_ab[-1]["hechos"]), "de 14")
ver(filas_ab, *AB)

**Lupa 1 · el corte beta en C2.** C2 (MAX) recibe 7, y $7 \ge \beta = 5$:
C (MIN) ya tiene 5 y nunca escogería algo que vale 7 o más. La hoja 8 no se
genera. C2 **devuelve 7 aunque vale 8**: es una cota («al menos 7»), no su
valor.

In [ ]:
k = next(f["n"] for f in filas_ab if f["evento"] == "corte beta")
lupa(filas_ab, k, *AB)

**Lupa 2 · el corte alfa en D.** D (MIN) recibe 2, y $2 \le \alpha = 5$:
R ya tiene 5 asegurado con `centro`, y D vale a lo más 2. La hoja 12 no se
genera.

In [ ]:
k = next(f["n"] for f in filas_ab if f["evento"] == "corte alfa")
lupa(filas_ab, k, *AB)

**Pruébalo.** Invierte el orden de los hijos en todo el árbol: ¿cuántos
nodos genera ahora alfa-beta? ¿Cambia el valor?

<details><summary>Qué pasa</summary>

Genera los 14: no corta nada, porque lo bueno llega al final y α sube
tarde. El valor sigue siendo 5 (por el mismo camino, que ahora se llama
`centro` igual). El orden cambia el trabajo, nunca el valor.

</details>

In [ ]:
def invertir(a):
    return tuple(map(invertir, a[::-1])) if isinstance(a, tuple) else a

usar(invertir(ARBOL_T))
filas, jugada = correr(decidir_alfa_beta("R"))
print("jugada:", jugada, "· generados:", len(filas[-1]["hechos"]))
usar(ARBOL_T)

## 6 · Minimax con corte

Si el árbol no cabe, se mira solo `d` jugadas adelante. Donde se acaba el
horizonte, el nodo no se expande: vale `EVAL(s)`, una estimación. Las hojas
de T son finales con $U$ = número/100, así que la línea 8 devuelve el
número.

In [ ]:
def decidir_con_corte(s, d):             # (1)
    mejor_valor = -inf                   # (2)
    mejor_jugada = None                  # (2)
    yield paso("inicio", "2", locals())
    for a in acciones(s):                # (3)
        h = transicion(s, a)             # (4) una jugada menos
        w = yield from minimax_con_corte(h, d - 1, [s, h])   # (4)
        if w > mejor_valor:              # (5) estricto
            mejor_valor = w
            mejor_jugada = a
        yield paso("raiz", "4–5", locals())
    yield paso("fin", "6", locals())
    return mejor_jugada                  # (6) jugada

In [ ]:
def minimax_con_corte(s, d, pila):       # (7) d: lo que queda
    if es_final(s):                      # (8) un final vale 100·U
        return 100 * u_final(s)
    if d == 0:                           # (9) nodo de corte
        return evaluar(s)
    if pl(s) == "MAX":                   # (10)
        return (yield from mcc_max(s, d, pila))
    else:                                # (16) MIN
        return (yield from mcc_min(s, d, pila))

In [ ]:
def mcc_max(s, d, pila):                 # la rama MAX
    v = -inf                             # (11)
    yield paso("entra", "11", locals())
    for a in acciones(s):                # (12)
        h = transicion(s, a)             # (13) una jugada menos
        w = yield from minimax_con_corte(h, d - 1, pila + [h])  # (13)
        v = max(v, w)                    # (14) mayor
        yield paso("regresa", "13–14", locals())
    return v                             # (15)

In [ ]:
def mcc_min(s, d, pila):                 # la rama MIN
    v = inf                              # (17)
    yield paso("entra", "17", locals())
    for a in acciones(s):                # (18)
        h = transicion(s, a)             # (19) una jugada menos
        w = yield from minimax_con_corte(h, d - 1, pila + [h])  # (19)
        v = min(v, w)                    # (20) menor
        yield paso("regresa", "19–20", locals())
    return v                             # (21)

In [ ]:
usar(ARBOL_T)
for d in (1, 2, 3):
    filas, jugada = correr(decidir_con_corte("R", d))
    print(f"d = {d}: {jugada:<6} con {fmt(filas[-1]['v'])},",
          f"{len(filas[-1]['hechos'])} nodos generados")
filas_c2, _ = correr(decidir_con_corte("R", 2))
ver(filas_c2, decidir_con_corte, minimax_con_corte, mcc_max, mcc_min)

Con `d = 1` cae en la trampa: `der` parece valer 7, pero D en realidad vale
2. Con `d = 2` ya ve la respuesta de MIN. En la figura, verde es `EVAL`.

**Pruébalo.** Con `d = 1`, sube `EVAL_T["C"]` a 8: ¿qué jugada sale?

<details><summary>Qué pasa</summary>

`centro`, con 8. Con el horizonte tan corto, la jugada la decide la
estimación, no el juego.

</details>

In [ ]:
EVAL_T["C"] = 4                          # cambia el 4
print("d = 1:", correr(decidir_con_corte("R", 1))[1])
EVAL_T["C"] = 4                          # la de la página

## 7 · Profundización iterativa: jugar contra el reloj

No sabes cuánto tiempo queda. Busca con `d = 1`, luego `d = 2`, `d = 3`…
y entrega la jugada de **la última búsqueda completa**. La jugada anterior
va primero: α sube pronto y se poda más.

In [ ]:
def alfa_beta_con_corte(s, d, alfa, beta, pila):   # (14)
    if es_final(s):                      # (15)
        return 100 * u_final(s)
    if d == 0:                           # (16) se estima
        return evaluar(s)
    if pl(s) == "MAX":                   # (17)
        return (yield from abc_max(s, d, alfa, beta, pila))
    else:                                # (25)
        return (yield from abc_min(s, d, alfa, beta, pila))

In [ ]:
def abc_max(s, d, alfa, beta, pila):     # la rama MAX
    v = -inf                             # (18)
    yield paso("entra", "18", locals())
    for a in acciones(s):                # (19)
        h = transicion(s, a)             # (20)
        w = yield from alfa_beta_con_corte(h, d - 1, alfa, beta, pila + [h])
        v = max(v, w)                    # (21)
        if v >= beta:                    # (22) corte beta
            yield paso("corte beta", "20–22", locals())
            yield from podar(s, a, pila, "beta")
            return v                     # (22)
        alfa = max(alfa, v)              # (23)
        yield paso("regresa", "20–23", locals())
    return v                             # (24)

In [ ]:
def abc_min(s, d, alfa, beta, pila):     # la rama MIN
    v = inf                              # (26)
    yield paso("entra", "26", locals())
    for a in acciones(s):                # (27)
        h = transicion(s, a)             # (28)
        w = yield from alfa_beta_con_corte(h, d - 1, alfa, beta, pila + [h])
        v = min(v, w)                    # (29)
        if v <= alfa:                    # (30) corte alfa
            yield paso("corte alfa", "28–30", locals())
            yield from podar(s, a, pila, "alfa")
            return v                     # (30)
        beta = min(beta, v)              # (31)
        yield paso("regresa", "28–31", locals())
    return v                             # (32)

El reloj es una función, `queda_tiempo()`. El de la página alcanza justo
para tres búsquedas completas. Con `RELOJ["nodos"] = N`, se acaba al
generar N nodos, contando todas las búsquedas.

In [ ]:
RELOJ = {"nodos": None}                  # None: el reloj de la página

def queda_tiempo():
    if RELOJ["nodos"] is None:
        return CUENTA["terminadas"] < 3  # tres búsquedas completas
    return CUENTA["total"] < RELOJ["nodos"]

In [ ]:
def una_busqueda(s, d, jugada):          # las líneas 5 a 10
    alfa, mejor_jugada = -inf, jugada    # (5)
    yield paso("inicio", "5", locals())
    resto = [a for a in acciones(s) if a != jugada]     # (6)
    for a in [jugada] + resto:           # (6) la anterior primero
        h = transicion(s, a)             # (7) le quedan d − 1
        w = yield from alfa_beta_con_corte(h, d - 1, alfa, inf, [s, h])
        if not queda_tiempo():           # (8) a medias: no sirve
            return True, jugada
        if w == 100:                     # (9) gana seguro
            return True, a
        if w > alfa:                     # (10) mejor que lo visto
            alfa, mejor_jugada = w, a
        yield paso("raiz", "7–10", locals())
    return False, mejor_jugada

In [ ]:
def profundizacion_iterativa(s):         # (1)
    jugada = acciones(s)[0]              # (2) siempre hay algo
    d = 1                                # (3)
    yield paso("antes", "2–3", locals())
    while queda_tiempo():                # (4)
        nueva_busqueda(s)                # la cuenta vuelve a R
        salir, nueva = yield from una_busqueda(s, d, jugada)
        if salir:                        # (8) o (9): se entrega ya
            return nueva
        jugada = nueva                   # (11) la búsqueda terminó
        yield paso("fin", "11–12", locals())
        d = d + 1                        # (12)
    yield paso("entrega", "13", locals())
    return jugada                        # (13)

In [ ]:
usar(ARBOL_T)
RELOJ["nodos"] = None
filas_pi, jugada = correr(profundizacion_iterativa("R"))
for f in filas_pi:
    if f["evento"] == "fin":
        print(f"d = {f['d']}: deja lista {f['jugada']:<6}",
              f"({len(f['hechos'])} nodos)")
print("se entrega:", jugada)
ver(filas_pi, profundizacion_iterativa, una_busqueda,
    alfa_beta_con_corte, abc_max, abc_min)

**El reloj de N nodos.** ¿Qué jugada se entrega si el tiempo alcanza para
generar solo N nodos?

In [ ]:
antes = None
for n in range(1, 27):
    RELOJ["nodos"] = n
    jugada = correr(profundizacion_iterativa("R"))[1]
    if jugada != antes:
        print(f"desde N = {n:>2}: se entrega {jugada}")
        antes = jugada
RELOJ["nodos"] = None

**Pruébalo.** Con un reloj de 12 nodos se entrega `der`, aunque la
búsqueda con `d = 2` ya había visto que `centro` vale más. ¿Por qué?

<details><summary>Respuesta</summary>

Porque esa búsqueda no terminó. Una búsqueda a medias puede no haber visto
la respuesta que deshace su jugada; se entrega la de la última búsqueda
completa, la de `d = 1`.

</details>

## 8 · Cierre

| algoritmo | jugada | nodos generados |
|---|---|---|
| minimax | centro (5) | 14 de 14 |
| expectiminimax (I, C, D de azar) | der (7) | 14 de 14 |
| alfa-beta en T1 / en T | izq (3) / centro (5) | 6 de 7 / 12 de 14 |
| con corte, d = 1, 2, 3 | der, centro, centro | 4, 10, 14 |
| profundización iterativa | centro | 4 + 10 + 11 |

La celda de abajo vuelve a calcular la tabla.

In [ ]:
corridas = [("minimax", ARBOL_T, (), lambda: decidir_minimax("R")),
            ("azar", ARBOL_T, "ICD", lambda: decidir_expectiminimax("R")),
            ("alfa-beta T1", ARBOL_T1, (), lambda: decidir_alfa_beta("R")),
            ("alfa-beta T", ARBOL_T, (), lambda: decidir_alfa_beta("R"))]
corridas += [(f"corte d = {d}", ARBOL_T, (),
              lambda d=d: decidir_con_corte("R", d)) for d in (1, 2, 3)]
for nombre, arbol, azar, algoritmo in corridas:
    usar(arbol, azar)
    filas, jugada = correr(algoritmo())
    print(f"{nombre:<13} {jugada:<7} {len(filas[-1]['hechos'])} nodos")
usar(ARBOL_T)

**Tres preguntas.**

1. ¿Por qué alfa-beta da la misma jugada que minimax aunque genere menos?
2. ¿Por qué expectiminimax elige `der` si minimax elige `centro`?
3. ¿Qué entrega la profundización iterativa si el reloj se acaba a mitad
   de `d = 3`?

<details><summary>Respuestas</summary>

1. Solo deja de mirar lo que no puede cambiar la decisión: un nodo podado
   está fuera de la ventana (α, β), y su padre ya tiene algo mejor.
2. Con azar, D ya no es «la peor hoja», sino el promedio: $(2 + 12)/2 = 7$.
3. `centro`, la jugada que dejó lista `d = 2`. La de `d = 3` está a medias.

</details>